# Notebook 2: Experiment 2 — Cross-Stock Prediction (80/20)
## A Comparative Analysis of BiLSTM and BiGRU for Stock Price Prediction

**Experiment:** Train on one stock's daily data, predict on another stock's daily test data.  
**Train/Test Split:** 80/20 (chronological)  
**Models:** BiLSTM, BiGRU, LSTM, GRU  
**Scaler:** ProportionScaler (÷ 10,501 BBCA ATH)  
**Metrics:** MSE, RMSE, MAE, MAPE, R² Score  


In [1]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '.')
from stock_prediction_utils import *

set_seed()
set_ieee_style()

DATA_DIR = 'dataset'

TRAIN_RATIO = 0.8
RATIO_LABEL = '80_20'
EXP_LABEL = f'Exp2_{RATIO_LABEL}'

os.makedirs(f'figures/{EXP_LABEL}', exist_ok=True)
os.makedirs(f'models/{EXP_LABEL}', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f"Experiment 2 - Cross-Stock Prediction (80/20)")


stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

GPU Memory allocation: Dynamic growth up to 95% (for optimal utilization)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
Experiment 2 - Cross-Stock Prediction (80/20)


In [2]:
# Load all daily data
print("Loading daily data...")
daily_data = load_all_daily_data(DATA_DIR)
print("\nAll daily data loaded!")


Loading daily data...
  TLKM: 5243 records, Date range: 2004-09-28 to 2025-12-31
  BBCA: 5244 records, Date range: 2004-09-28 to 2025-12-31
  ASII: 5244 records, Date range: 2004-09-28 to 2025-12-31
  UNVR: 5245 records, Date range: 2004-09-28 to 2025-12-31

All daily data loaded!


## Run All Cross-Stock Experiments

### Methodology: Zero-Shot Transfer Learning
Instead of retraining from scratch, we load pre-trained models from **Experiment 1** (with same-stock data) and directly apply them to predict different stocks' test data. This tests how well models generalize across stocks without any domain adaptation.

**Expected result:** Performance will likely be worse than same-stock predictions due to different price ranges, volatility, and patterns across stocks. However, this shows raw transfer capability.

In [3]:
# ============================================================
# EXPERIMENT 2: Cross-stock prediction (using Exp1 pre-trained models)
# Load models trained on Stock A, test on Stock B (zero-shot transfer)
# ============================================================
from tensorflow.keras.models import load_model

all_results = []
all_predictions = {}  # {(train_stock, test_stock): {model_type: (y_true, y_pred, dates)}}

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue  # Skip same-stock (covered in Exp 1)
        
        pair_key = (train_stock, test_stock)
        print(f"\n{'#'*60}")
        print(f"# TRAIN: {train_stock} -> TEST: {test_stock}")
        print(f"# Using pre-trained Exp1 models (zero-shot transfer)")
        print(f"{'#'*60}")
        
        # Prepare cross-stock data
        X_train, y_train, X_test, y_test, test_dates = prepare_cross_stock_data(
            daily_data[train_stock], daily_data[test_stock],
            train_ratio=TRAIN_RATIO, lookback=LOOKBACK
        )
        print(f"  X_train: {X_train.shape}, X_test: {X_test.shape}")
        
        all_predictions[pair_key] = {}
        
        for model_type in MODEL_TYPES:
            # Build Exp1 model filename
            exp1_model_path = f'models/Exp1_80_20/Exp1_80_20_{train_stock}_{model_type}_best.keras'
            
            if not os.path.exists(exp1_model_path):
                print(f"  ⚠️  Model not found: {exp1_model_path}")
                continue
            
            print(f"\n  Loading {model_type} from: {exp1_model_path}")
            
            # Load pre-trained model
            model = load_model(exp1_model_path)
            
            # Make predictions on test data (NO training)
            y_pred_scaled = model.predict(X_test, verbose=0).flatten()
            
            # Inverse scale to original values
            y_true_inv = proportion_inverse_scale(y_test)
            y_pred_inv = proportion_inverse_scale(y_pred_scaled)
            
            # Evaluate
            metrics = evaluate_predictions(y_true_inv, y_pred_inv)
            
            result = {
                'Train_Stock': train_stock,
                'Test_Stock': test_stock,
                'Model': model_type,
                **metrics
            }
            all_results.append(result)
            all_predictions[pair_key][model_type] = (y_true_inv, y_pred_inv, test_dates)
            
            print(f"    RMSE: {metrics['RMSE']:.4f}, MAE: {metrics['MAE']:.4f}, R²: {metrics['R2']:.6f}")
            
            # Plot prediction
            plot_actual_vs_predicted(
                test_dates, y_true_inv, y_pred_inv,
                model_type, f'Train_{train_stock}_Test_{test_stock}',
                EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
            )

print("\n\nAll Experiment 2 (80/20) cross-stock prediction complete!")



############################################################
# TRAIN: TLKM -> TEST: BBCA
# Using pre-trained Exp1 models (zero-shot transfer)
############################################################
  X_train: (4193, 1, 1), X_test: (1049, 1, 1)

  Loading BiLSTM from: models/Exp1_80_20/Exp1_80_20_TLKM_BiLSTM_best.keras
    RMSE: 290.2392, MAE: 255.8343, R²: 0.922062
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_TLKM_Test_BBCA_BiLSTM_prediction.png

  Loading BiGRU from: models/Exp1_80_20/Exp1_80_20_TLKM_BiGRU_best.keras
    RMSE: 192.7421, MAE: 159.8723, R²: 0.965629
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_TLKM_Test_BBCA_BiGRU_prediction.png

  Loading LSTM from: models/Exp1_80_20/Exp1_80_20_TLKM_LSTM_best.keras
    RMSE: 388.5765, MAE: 355.2503, R²: 0.860302
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_TLKM_Test_BBCA_LSTM_prediction.png

  Loading GRU from: models/Exp1_80_20/Exp1_80_20_TLKM_GRU_best.keras
    RMSE: 326.4622, MAE: 295.3493, R²: 0.901394
  

## Results Summary

In [4]:
# ============================================================
# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(all_results)
print_results_table(results_df, f"Experiment 2 - Cross-Stock Prediction (80/20)")

results_df.to_csv(f'results/{EXP_LABEL}_results.csv', index=False)
print(f"Results saved to results/{EXP_LABEL}_results.csv")



  Experiment 2 - Cross-Stock Prediction (80/20)
Train_Stock Test_Stock  Model         MSE     RMSE      MAE  MAPE (%)       R2
       TLKM       BBCA BiLSTM  84238.7958 290.2392 255.8343    3.0310 0.922062
       TLKM       BBCA  BiGRU  37149.5335 192.7421 159.8723    1.9198 0.965629
       TLKM       BBCA   LSTM 150991.7317 388.5765 355.2503    4.2131 0.860302
       TLKM       BBCA    GRU 106577.5592 326.4622 295.3493    3.5164 0.901394
       TLKM       ASII BiLSTM   9737.2194  98.6774  74.0364    1.5424 0.973645
       TLKM       ASII  BiGRU   8859.7362  94.1262  70.3550    1.4767 0.976020
       TLKM       ASII   LSTM  13484.4150 116.1224  90.3141    1.8654 0.963502
       TLKM       ASII    GRU  12600.0544 112.2500  87.0225    1.8037 0.965896
       TLKM       UNVR BiLSTM   4938.6495  70.2755  47.7412    1.7568 0.994323
       TLKM       UNVR  BiGRU   4986.1021  70.6123  48.3128    1.7924 0.994268
       TLKM       UNVR   LSTM   5301.8598  72.8139  50.4723    1.8540 0.993905
   

## Visualizations

In [5]:
# Reload the module to get the latest fixes
import importlib
import stock_prediction_utils
importlib.reload(stock_prediction_utils)
from stock_prediction_utils import *
print("✓ Module reloaded successfully")

stock_prediction_utils.py loaded successfully!
  ProportionScaler max value: 10501.0
  Lookback: 1, Epochs: 100, Batch size: 64
  Architecture: 2 layers, 64 units, dropout=0.2
  Stocks: ['TLKM', 'BBCA', 'ASII', 'UNVR']
  Models: ['BiLSTM', 'BiGRU', 'LSTM', 'GRU']

GPU SETUP - CUDA Available
Number of GPUs detected: 1
  GPU 0: /physical_device:GPU:0

GPU Memory allocation: Dynamic growth up to 95% (for optimal utilization)
TensorFlow configured to use GPU


Device Configuration:
  GPUs available: 1
  CPUs available: 1
  TensorFlow will use GPU for computations
✓ Module reloaded successfully


In [6]:
# ============================================================
# INTERACTIVE RESULTS VISUALIZATIONS - CROSS-STOCK
# ============================================================

print("Generating interactive results visualizations...\n")

# 1. Cross-Stock Results Dashboard
print("1. Generating Cross-Stock Results Dashboard...")
fig1, html1 = create_interactive_results_dashboard_exp2(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html1}")
fig1.show()

print()

# 2. Transfer Learning Matrix Heatmaps
print("2. Generating Transfer Learning Matrices...")
for metric in ['RMSE', 'MAE', 'R2']:
    try:
        create_interactive_transfer_matrix(
            results_df, metric, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )
        print(f"   ✓ {metric} transfer matrices generated")
    except Exception as e:
        print(f"   ⚠ Skipping {metric}: {str(e)}")

# Display one example
fig_example, html_example = create_interactive_transfer_matrix(
    results_df, 'RMSE', EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print("   (Displaying BiLSTM transfer matrix example)\n")

print()

# 3. Metrics Comparison Chart
print("3. Generating Metrics Comparison Chart...")
fig3, html3 = create_interactive_metrics_comparison(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}',
    metrics=['RMSE', 'MAE', 'R2']
)
print(f"   ✓ Saved: {html3}")
fig3.show()

print()

# 4. Model Radar Chart
print("4. Generating Model Radar Chart...")
fig4, html4 = create_interactive_model_radar_chart(
    results_df, EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
)
print(f"   ✓ Saved: {html4}")
fig4.show()

print("\n✓ All interactive results visualizations generated successfully!")

print("\n" + "="*70)

Generating interactive results visualizations...

1. Generating Cross-Stock Results Dashboard...
   ✓ Saved: figures/Exp2_80_20/Exp2_80_20_crossstock_dashboard.html



2. Generating Transfer Learning Matrices...
   ✓ RMSE transfer matrices generated
   ✓ MAE transfer matrices generated
   ✓ R2 transfer matrices generated
   (Displaying BiLSTM transfer matrix example)


3. Generating Metrics Comparison Chart...
   ✓ Saved: figures/Exp2_80_20/Exp2_80_20_metrics_comparison.html



4. Generating Model Radar Chart...
   ✓ Saved: figures/Exp2_80_20/Exp2_80_20_model_radar.html



✓ All interactive results visualizations generated successfully!



### Interactive Results Visualizations

In [7]:
# ============================================================
# HEATMAPS PER MODEL
# ============================================================
for metric in ['RMSE', 'MAE', 'MAPE (%)', 'R2']:
    plot_metrics_heatmap(
        results_df, metric, EXP_LABEL,
        row_col='Train_Stock', col_col='Test_Stock',
        save_dir=f'figures/{EXP_LABEL}'
    )

print("All heatmaps saved!")


  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiLSTM_RMSE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiGRU_RMSE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_LSTM_RMSE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_GRU_RMSE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiLSTM_MAE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiGRU_MAE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_LSTM_MAE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_GRU_MAE_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiLSTM_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiGRU_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_LSTM_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_GRU_MAPE_pct_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiLSTM_R2_heatmap.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_BiGRU_R2_heatmap.png
  Figure saved: figures/Exp2_80

In [8]:
# ============================================================
# COMPARISON: All models for each train->test pair
# ============================================================
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        preds = {mt: all_predictions[pair_key][mt][1] for mt in MODEL_TYPES if mt in all_predictions[pair_key]}
        
        plot_all_models_comparison(
            dates, y_true, preds,
            f'Train_{train_stock}_Test_{test_stock}',
            EXP_LABEL, save_dir=f'figures/{EXP_LABEL}'
        )

print("All comparison plots saved!")


  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_TLKM_Test_BBCA_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_TLKM_Test_ASII_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_TLKM_Test_UNVR_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_BBCA_Test_TLKM_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_BBCA_Test_ASII_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_BBCA_Test_UNVR_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_ASII_Test_TLKM_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_ASII_Test_BBCA_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_ASII_Test_UNVR_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_UNVR_Test_TLKM_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_UNVR_Test_BBCA_all_models.png
  Figure saved: figures/Exp2_80_20/Exp2_80_20_Train_UNVR_Test_ASII_all_models.png
All comparison p

In [9]:
# ============================================================
# SUMMARY: BEST MODEL PER CROSS-STOCK PAIR
# ============================================================
print("\n" + "="*70)
print("  BEST MODEL PER PAIR (by RMSE)")
print("="*70)
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_data = results_df[
            (results_df['Train_Stock'] == train_stock) &
            (results_df['Test_Stock'] == test_stock)
        ]
        if pair_data.empty:
            continue
        best_idx = pair_data['RMSE'].idxmin()
        best = pair_data.loc[best_idx]
        print(f"  {train_stock} -> {test_stock}: {best['Model']} "
              f"(RMSE={best['RMSE']:.4f}, R²={best['R2']:.6f})")



  BEST MODEL PER PAIR (by RMSE)
  TLKM -> BBCA: BiGRU (RMSE=192.7421, R²=0.965629)
  TLKM -> ASII: BiGRU (RMSE=94.1262, R²=0.976020)
  TLKM -> UNVR: BiLSTM (RMSE=70.2755, R²=0.994323)
  BBCA -> TLKM: LSTM (RMSE=60.1657, R²=0.978811)
  BBCA -> ASII: BiGRU (RMSE=82.4868, R²=0.981584)
  BBCA -> UNVR: LSTM (RMSE=73.5222, R²=0.993786)
  ASII -> TLKM: GRU (RMSE=54.8323, R²=0.982401)
  ASII -> BBCA: GRU (RMSE=147.5185, R²=0.979866)
  ASII -> UNVR: GRU (RMSE=69.3781, R²=0.994467)
  UNVR -> TLKM: BiLSTM (RMSE=57.5160, R²=0.980636)
  UNVR -> BBCA: GRU (RMSE=123.7209, R²=0.985838)
  UNVR -> ASII: LSTM (RMSE=81.8216, R²=0.981879)


## Interactive Visualization (Plotly)

In [13]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ============================================================
# INTERACTIVE VISUALIZATION - ALL MODELS COMPARISON (Plotly)
# Actual Prices in RED (#FF0000), Models in Model Colors
# ============================================================
print("Generating interactive plots...")

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        print(f"  Generating interactive plot for {train_stock} → {test_stock}...")
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        
        # Create figure with all models visible
        fig = go.Figure()
        
        # Linestyles for variety
        linestyles_map = {
            'BiLSTM': 'solid',
            'BiGRU': 'dash',
            'LSTM': 'dot',
            'GRU': 'dashdot'
        }
        
        # Add actual values in RED (always visible)
        fig.add_trace(go.Scatter(
            x=dates, y=y_true,
            name='Actual Price',
            mode='lines',
            line=dict(color='#FF0000', width=2.0, dash='solid'),
            hovertemplate='<b>ACTUAL PRICE</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
            visible=True,
            opacity=0.95
        ))
        
        # Add all model predictions
        for model_type in MODEL_TYPES:
            if model_type in all_predictions[pair_key]:
                y_pred = all_predictions[pair_key][model_type][1]
                
                fig.add_trace(go.Scatter(
                    x=dates, y=y_pred,
                    name=f'{model_type} (Predicted)',
                    mode='lines',
                    line=dict(
                        color=MODEL_COLORS[model_type],
                        width=2.0,
                        dash=linestyles_map.get(model_type, 'solid')
                    ),
                    hovertemplate=f'<b>{model_type}</b><br>Date: %{{x|%Y-%m-%d}}<br>Price: IDR %{{y:,.2f}}<extra></extra>',
                    visible=True,
                    opacity=0.85
                ))
        
        # Update layout
        fig.update_layout(
            title=f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>Zero-Shot Transfer Learning (80/20 Split)</sub>',
            xaxis_title='Date',
            yaxis_title='Close Price (IDR)',
            hovermode='x unified',
            template='plotly_white',
            font=dict(size=40, family="Times New Roman"),
            height=700,
            width=1200,
            margin=dict(l=80, r=80, t=120, b=80),
            xaxis=dict(
                gridwidth=1,
                gridcolor='lightgray',
                tickfont=dict(size=30, family="Times New Roman")
            ),
            yaxis=dict(
                gridwidth=1,
                gridcolor='lightgray',
                tickfont=dict(size=30, family="Times New Roman")
            ),
            legend=dict(
                x=0.99,
                y=0.99,
                xanchor="right",
                yanchor="top",
                bgcolor="rgba(255, 255, 255, 0.9)",
                bordercolor="gray",
                borderwidth=1,
                font=dict(size=20, family="Times New Roman")
            ),
            xaxis_title_font=dict(size=40, family="Times New Roman"),
            yaxis_title_font=dict(size=40, family="Times New Roman")
        )
        
        # Add range slider
        fig.update_xaxes(rangeslider_visible=False)
        
        # Save ALL MODELS plot
        html_filename = f'figures/{EXP_LABEL}/interactive_{train_stock}_{test_stock}_all_models.html'
        fig.write_html(html_filename, config=PLOTLY_HTML_CONFIG)
        print(f"    ✓ Saved: {html_filename}")

print("\n✓ All interactive plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*.html")
print(f"  Actual prices displayed in RED (#FF0000)")
print(f"  Font: Times New Roman, Size: 20")

Generating interactive plots...
  Generating interactive plot for TLKM → BBCA...
    ✓ Saved: figures/Exp2_80_20/interactive_TLKM_BBCA_all_models.html
  Generating interactive plot for TLKM → ASII...
    ✓ Saved: figures/Exp2_80_20/interactive_TLKM_ASII_all_models.html
  Generating interactive plot for TLKM → UNVR...
    ✓ Saved: figures/Exp2_80_20/interactive_TLKM_UNVR_all_models.html
  Generating interactive plot for BBCA → TLKM...
    ✓ Saved: figures/Exp2_80_20/interactive_BBCA_TLKM_all_models.html
  Generating interactive plot for BBCA → ASII...
    ✓ Saved: figures/Exp2_80_20/interactive_BBCA_ASII_all_models.html
  Generating interactive plot for BBCA → UNVR...
    ✓ Saved: figures/Exp2_80_20/interactive_BBCA_UNVR_all_models.html
  Generating interactive plot for ASII → TLKM...
    ✓ Saved: figures/Exp2_80_20/interactive_ASII_TLKM_all_models.html
  Generating interactive plot for ASII → BBCA...
    ✓ Saved: figures/Exp2_80_20/interactive_ASII_BBCA_all_models.html
  Generating int

In [11]:
# ============================================================
# INTERACTIVE VISUALIZATION - TOGGLE BUTTONS (Plotly)
# Actual Prices in RED (#FF0000)
# ============================================================
print("Generating interactive toggle plots...")

for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        
        print(f"  Generating toggle plot for {train_stock} → {test_stock}...")
        
        y_true = all_predictions[pair_key][MODEL_TYPES[0]][0]
        dates = all_predictions[pair_key][MODEL_TYPES[0]][2]
        
        # Create figure with toggle buttons
        fig = go.Figure()
        
        linestyles_map = {
            'BiLSTM': 'solid',
            'BiGRU': 'dash',
            'LSTM': 'dot',
            'GRU': 'dashdot'
        }
        
        # Add actual values in RED (always visible)
        fig.add_trace(go.Scatter(
            x=dates, y=y_true,
            name='Actual Price',
            mode='lines',
            line=dict(color='#FF0000', width=2.0, dash='solid'),
            hovertemplate='<b>ACTUAL PRICE</b><br>Date: %{x|%Y-%m-%d}<br>Price: IDR %{y:,.2f}<extra></extra>',
            visible=True,
            opacity=0.95
        ))
        
        # Add each model with visibility control
        for idx, model_type in enumerate(MODEL_TYPES):
            if model_type in all_predictions[pair_key]:
                y_pred = all_predictions[pair_key][model_type][1]
                
                # First model visible by default, others hidden
                is_visible = True if idx == 0 else False
                
                fig.add_trace(go.Scatter(
                    x=dates, y=y_pred,
                    name=f'{model_type} (Predicted)',
                    mode='lines',
                    line=dict(
                        color=MODEL_COLORS[model_type],
                        width=2.0,
                        dash=linestyles_map.get(model_type, 'solid')
                    ),
                    hovertemplate=f'<b>{model_type}</b><br>Date: %{{x|%Y-%m-%d}}<br>Price: IDR %{{y:,.2f}}<extra></extra>',
                    visible=is_visible,
                    opacity=0.85
                ))
        
        # Create buttons for toggling models
        buttons = []
        for i, model_type in enumerate(MODEL_TYPES):
            # Create visibility list: [True for Actual, False for all models except this one, True for this model]
            visibility = [True] + [j == i for j in range(len(MODEL_TYPES))]
            
            buttons.append(
                dict(
                    label=model_type,
                    method='update',
                    args=[
                        {'visible': visibility},
                        {'title': f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>{model_type} Model (80/20 Split)</sub>'}
                    ]
                )
            )
        
        # Add "Show All" button
        buttons.insert(0, dict(
            label='All Models',
            method='update',
            args=[
                {'visible': [True] * (len(MODEL_TYPES) + 1)},
                {'title': f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>All Models Comparison (80/20 Split)</sub>'}
            ]
        ))
        
        # Update layout with buttons
        fig.update_layout(
            updatemenus=[
                dict(
                    active=0,
                    buttons=buttons,
                    direction="down",
                    pad={"r": 10, "t": 10},
                    showactive=True,
                    x=0.01,
                    xanchor="left",
                    y=0.99,
                    yanchor="top",
                    bgcolor="rgba(200, 200, 200, 0.9)",
                    bordercolor="gray",
                    borderwidth=2,
                    font=dict(size=16, family="Times New Roman")
                )
            ],
            title=f'<b>Cross-Stock Prediction: {train_stock} → {test_stock}</b><br><sub>All Models Comparison (80/20 Split)</sub>',
            xaxis_title='Date',
            yaxis_title='Close Price (IDR)',
            hovermode='x unified',
            template='plotly_white',
            font=dict(size=20, family="Times New Roman"),
            height=700,
            width=1200,
            margin=dict(l=80, r=80, t=150, b=80),
            xaxis=dict(
                gridwidth=1,
                gridcolor='lightgray',
                tickfont=dict(size=20, family="Times New Roman")
            ),
            yaxis=dict(
                gridwidth=1,
                gridcolor='lightgray',
                tickfont=dict(size=20, family="Times New Roman")
            ),
            legend=dict(
                x=0.99,
                y=0.99,
                xanchor="right",
                yanchor="top",
                bgcolor="rgba(255, 255, 255, 0.9)",
                bordercolor="gray",
                borderwidth=1,
                font=dict(size=18, family="Times New Roman")
            ),
            xaxis_title_font=dict(size=25, family="Times New Roman"),
            yaxis_title_font=dict(size=25, family="Times New Roman")
        )
        
        # Save TOGGLE plot
        html_filename = f'figures/{EXP_LABEL}/interactive_{train_stock}_{test_stock}_toggle.html'
        fig.write_html(html_filename, config=PLOTLY_HTML_CONFIG)
        print(f"    ✓ Saved: {html_filename}")

print("\n✓ All interactive toggle plots generated and saved!")
print(f"  Location: figures/{EXP_LABEL}/interactive_*_toggle.html")
print(f"  Actual prices displayed in RED (#FF0000)")
print(f"  Font: Times New Roman, Size: 20")

Generating interactive toggle plots...
  Generating toggle plot for TLKM → BBCA...
    ✓ Saved: figures/Exp2_80_20/interactive_TLKM_BBCA_toggle.html
  Generating toggle plot for TLKM → ASII...
    ✓ Saved: figures/Exp2_80_20/interactive_TLKM_ASII_toggle.html
  Generating toggle plot for TLKM → UNVR...
    ✓ Saved: figures/Exp2_80_20/interactive_TLKM_UNVR_toggle.html
  Generating toggle plot for BBCA → TLKM...
    ✓ Saved: figures/Exp2_80_20/interactive_BBCA_TLKM_toggle.html
  Generating toggle plot for BBCA → ASII...
    ✓ Saved: figures/Exp2_80_20/interactive_BBCA_ASII_toggle.html
  Generating toggle plot for BBCA → UNVR...
    ✓ Saved: figures/Exp2_80_20/interactive_BBCA_UNVR_toggle.html
  Generating toggle plot for ASII → TLKM...
    ✓ Saved: figures/Exp2_80_20/interactive_ASII_TLKM_toggle.html
  Generating toggle plot for ASII → BBCA...
    ✓ Saved: figures/Exp2_80_20/interactive_ASII_BBCA_toggle.html
  Generating toggle plot for ASII → UNVR...
    ✓ Saved: figures/Exp2_80_20/inter

## Case-by-Case Interactive Actual vs Predicted
One interactive Plotly chart per notebook with a dropdown that walks through every test case. Each case shows the Actual price (red) plus all four model predictions, with a metrics panel (MSE, RMSE, MAE, MAPE, R²) for that case.

In [12]:
# ============================================================
# CASE-BY-CASE INTERACTIVE ACTUAL VS PREDICTED (Experiment 2)
# ============================================================
# Cases follow the test plan: 12 train -> test cross-stock pairs.
from collections import OrderedDict

cases_dict = OrderedDict()
for train_stock in STOCKS:
    for test_stock in STOCKS:
        if train_stock == test_stock:
            continue
        pair_key = (train_stock, test_stock)
        if pair_key not in all_predictions:
            continue
        available_mts = [mt for mt in MODEL_TYPES if mt in all_predictions[pair_key]]
        if not available_mts:
            continue
        y_true, _, dates = all_predictions[pair_key][available_mts[0]]
        predictions = {mt: all_predictions[pair_key][mt][1] for mt in available_mts}

        metrics = {}
        for mt in MODEL_TYPES:
            row = results_df[(results_df['Train_Stock'] == train_stock) &
                             (results_df['Test_Stock']  == test_stock) &
                             (results_df['Model']       == mt)]
            if not row.empty:
                r = row.iloc[0]
                metrics[mt] = {
                    'MSE'      : r.get('MSE'),
                    'RMSE'     : r.get('RMSE'),
                    'MAE'      : r.get('MAE'),
                    'MAPE (%)' : r.get('MAPE (%)'),
                    'R2'       : r.get('R2'),
                }

        label = f'{train_stock}→{test_stock}'
        cases_dict[label] = {
            'description' : f'Train {train_stock} Daily → Predict {test_stock} Daily',
            'dates'       : dates,
            'y_true'      : y_true,
            'predictions' : predictions,
            'metrics'     : metrics,
        }

ratio_pretty = RATIO_LABEL.replace('_', '/')
fig_case, html_case = create_case_by_case_actual_vs_predicted(
    cases_dict,
    experiment_title=f'Experiment 2: Cross-Stock Prediction ({ratio_pretty})',
    experiment_label=EXP_LABEL,
    save_dir=f'figures/{EXP_LABEL}',
)
print(f"✓ Case-by-case visualization saved: {html_case}")
print(f"  Cases ({len(cases_dict)}): {list(cases_dict.keys())}")
fig_case.show()


✓ Case-by-case visualization saved: figures/Exp2_80_20/Exp2_80_20_case_by_case_actual_vs_predicted.html
  Cases (12): ['TLKM→BBCA', 'TLKM→ASII', 'TLKM→UNVR', 'BBCA→TLKM', 'BBCA→ASII', 'BBCA→UNVR', 'ASII→TLKM', 'ASII→BBCA', 'ASII→UNVR', 'UNVR→TLKM', 'UNVR→BBCA', 'UNVR→ASII']
